Import Spark

In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("dic").getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
25/05/12 23:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4049. Attempting port 4050.
25/05/12 2

PySpark ML imports

In [21]:
from pyspark.ml.feature import RegexTokenizer, CountVectorizer, StopWordsRemover, IDF, StringIndexer, ChiSqSelector, Normalizer, VarianceThresholdSelector
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import OneVsRest, LinearSVC
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

Load the dataset into a DataFrame

In [2]:
df = spark.read.json("hdfs:///user/dic25_shared/amazon-reviews/full/reviews_devset.json")
df_sampled = df.sample(withReplacement=False, fraction=0.01, seed=42) # Downsample the devset

Define a custom regular expression for tokenization
This will split on whitespace, tabs, digits, and special characters you specified

In [3]:
regex_tokenizer = RegexTokenizer(
    inputCol="reviewText",
    outputCol="words",
    pattern=r"[\s\t\d\(\)\[\]\{\}\.\!\?,;:+=\-_'`~#@&*%€$§\\/]+"
)

Remove stopwords

In [4]:
with open("stopwords.txt") as f:
    stopwords = [line.strip() for line in f]
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words", stopWords=stopwords)

Apply CountVectorizer

In [5]:
cv = CountVectorizer(inputCol="filtered_words", outputCol="raw_features") # create a vector of frequency for each word (term)

Apply IDF

In [6]:
idf = IDF(inputCol="raw_features", outputCol="features")

Convert the `category` column to a numeric label

In [7]:
indexer = StringIndexer(inputCol="category", outputCol="category_index")

Apply chi-square feature selection for 2000 and 500 most relevant features

In [8]:
selector_2000 = ChiSqSelector(numTopFeatures=2000, # take top 2000 most relevant features
                         featuresCol="features", # use tf-idf vectors
                         outputCol="selected_features", # create new column
                         labelCol="category_index")

Add normalizer for the features

In [9]:
normalizer = Normalizer(inputCol="selected_features",
                        outputCol="normalized_features",
                        p=2.0)

Create the SVM classifier

In [10]:
svm = LinearSVC(featuresCol="normalized_features",
                labelCol="category_index",
                maxIter=10,
                regParam=0.1,
                standardization=True)

Create OneVsRest for multi-class classification

In [11]:
ovr = OneVsRest(classifier=svm,
                labelCol="category_index",
                featuresCol="normalized_features")

Evaluate the model

In [12]:
evaluator = MulticlassClassificationEvaluator(
    labelCol='category_index',
    predictionCol='prediction',
    metricName='f1'
)

Apply grid search for 3 regParams, 2 standardizations and 2 iterations

In [13]:
paramGrid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(svm.standardization, [True, False]) \
    .addGrid(svm.maxIter, [10, 20]) \
    .build()

Apply cross-validator with 3-folds

In [14]:
crossvalidation = CrossValidator(estimator=None,
                                 estimatorParamMaps=paramGrid,
                                 evaluator=evaluator,
                                 numFolds=3,
                                 seed=42)

Split the data into train, validation and test sets

In [15]:
train, val, test = df_sampled.randomSplit([0.7, 0.15, 0.15], seed=42) # split the data and set the random set to ensure reproducibility

Combine all stages into a pipeline with 2000 features

In [16]:
pipeline_2000 = Pipeline(stages=[regex_tokenizer,
                            stopwords_remover,
                            cv,
                            idf,
                            indexer,
                            selector_2000,
                            normalizer,
                            ovr])

In [17]:
crossvalidation.setEstimator(pipeline_2000)
cv_model_2000 = crossvalidation.fit(train)
predictions_2000 = cv_model_2000.transform(test)
f1_score_2000 = evaluator.evaluate(predictions_2000)

25/05/12 23:18:29 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/05/12 23:18:29 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
25/05/12 23:18:29 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/05/12 23:18:29 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


In [18]:
print(f"F1 score with 2000 features: {f1_score_2000}")

F1 score with 2000 features: 0.24547722578244854


Print F1 score for each hyperparameter combination

In [19]:
print("\nDetailed F1 scores for each parameter combination:")
for avg, params in zip(cv_model_2000.avgMetrics, cv_model_2000.getEstimatorParamMaps()):
    reg = params[svm.regParam]
    std = params[svm.standardization]
    iters = params[svm.maxIter]
    print(f"regParam={reg}, standardization={std}, maxIter={iters} --> F1 = {avg:.4f}")


Detailed F1 scores for each parameter combination:
regParam=0.01, standardization=True, maxIter=10 --> F1 = 0.2867
regParam=0.01, standardization=True, maxIter=20 --> F1 = 0.2900
regParam=0.01, standardization=False, maxIter=10 --> F1 = 0.1605
regParam=0.01, standardization=False, maxIter=20 --> F1 = 0.2938
regParam=0.1, standardization=True, maxIter=10 --> F1 = 0.3101
regParam=0.1, standardization=True, maxIter=20 --> F1 = 0.3122
regParam=0.1, standardization=False, maxIter=10 --> F1 = 0.1391
regParam=0.1, standardization=False, maxIter=20 --> F1 = 0.2897
regParam=1.0, standardization=True, maxIter=10 --> F1 = 0.2973
regParam=1.0, standardization=True, maxIter=20 --> F1 = 0.2862
regParam=1.0, standardization=False, maxIter=10 --> F1 = 0.1391
regParam=1.0, standardization=False, maxIter=20 --> F1 = 0.0424


Pipeline with `VarianceThresholdSelector`

In [ ]:
variance_threshold_selector = VarianceThresholdSelector(
    featuresCol ="features", 
    outputCol ="selected_features", 
    varianceThreshold = 0.01
)

pipeline_variance = Pipeline(stages=[
    regex_tokenizer,
    stopwords_remover,
    cv,
    idf,
    indexer,
    variance_threshold_selector,
    normalizer,
    ovr
])

In [ ]:
crossvalidation.setEstimator(pipeline_variance)
cv_model_variance = crossvalidation.fit(train)
predictions_variance = cv_model_variance.transform(test)
f1_score_variance = evaluator.evaluate(predictions_variance)

In [ ]:
print(f"F1 score with VarianceThresholdSelector: {f1_score_variance}")

Print F1 score for each hyperparameter combination

In [ ]:
print("\nDetailed F1 scores for each parameter combination with VarianceThresholdSelector:")
for avg, params in zip(cv_model_variance.avgMetrics, cv_model_variance.getEstimatorParamMaps()):
    reg = params[svm.regParam]
    std = params[svm.standardization]
    iters = params[svm.maxIter]
    print(f"regParam={reg}, standardization={std}, maxIter={iters} --> F1 = {avg:.4f}")